# SetReg-Net — Direct Set-Regression AoA/AoD (no heatmap, no blob detector)

Ei notebook base paper-er **image-to-image (256x256 heatmap)** formulation-take pura
badle diyeche. Baseline (UNet/ResNet/PIA-Net) shob-i input-take ekta 2-D heatmap-e
convert kore, tarpor blob detect kore angle ber kore. Ei notebook shorashori:

```
16x16x2 raw observation  ->  [model]  ->  L_max=9-ta "query", protita:
                                            - presence (etay ekta path ache ki na)
                                            - (sin psi, cos psi)
                                            - (sin phi, cos phi)
```

**Kono 256x256 decoder nei, kono blob detector nei, kono pixel quantization nei.**
DETR-er ("object detection as set prediction", Carion et al.) idea ধার kora --
Hungarian matching diye variable-shonkha-r path-ke fixed-shonkha-r query-r shathe
match kore train kora hoy.

## Ei redesign-er karon (mapa, onuman na)

Age-r PIA-Net-e (image-heatmap) tinta bug fix korar por-o dekha gelo:
- compute-er **80%** shudhu 32->256 upsampling decoder-e jachchilo
- output grid pixel-quantization-e ekta hard limit tairi korchilo
- blob detector clutter toiri korto (mean 8.3 blob, true L=3)

Ei tinta shomossha-i **structurally** ei notebook-e thakte pare na, karon kono
spatial upsampling-i nei.

## Guruttopurno: tulonar mapkathi (yardstick) BODLACHCHI NA

Model-er bhitorer kaj kora-r poddhoti shopurno notun, kintu:
- **Test set** : same Kaggle dataset, same L=3/P=16 condition
- **Metric**   : same RMSE-within-1-degree, same Pd (repo-r `TVT_Blob_Inference.py`-r
  `prepare_for_metric`/`get_ang_difference`/`filter_angles` hubohu reuse kora)
- **Reference**: same hardcoded UNet/ResNet number (repo-r committed reproduction)

Shudhu model output-ta shorashori angle bole, tai blob-detection step-ta baad --
eta pipeline-ke CHHOTO kore, alada metric banay na.

## Honest expectation

Ei architecture-er kono verified result nei ei mohuurte. Choto-scale probe (4800
sample, 300 step) e dekha geche gradient thik-i flow kore (loss 90%+ kome), kintu
oracle-selection Pd matro 3.4% -- mane matro etuku data-te kono architecture-i valo
korte parto na. Full-scale training-i asol uttor debe.

## Part 0 — Setup + imports

In [ ]:
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
        print(f'{import_name}: already available, skip.')
    except ImportError:
        print(f'{import_name}: not found, installing {pip_name} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

for pip_name, import_name in [('opencv-python-headless', 'cv2')]:
    ensure(pip_name, import_name)
print('Dependency check done.')

In [ ]:
import os, math, random, pickle, time

import numpy as np
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import cv2

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras import layers as L

# eager training loop (Part 6) calls the model fresh every step (no @tf.function
# graph caching like model.fit uses), so this benign complex64->float32 cast
# warning inside LearnedISTA would otherwise print on EVERY step -- silence it here.
tf.get_logger().setLevel('ERROR')
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Part 1 — Exact physics functions (PIA-Net-v2 theke hubohu, Bug 2 fix shoho)

`ev()`-er sign ar DFT-based codebook construction -- repo-r `dldoa_dataset_generation.py`
theke exact copy, jate physics dictionary data generation-er shathe mile।

In [ ]:
def wrapTo2Pi(x):
    return np.mod(x, 2 * np.pi)

def ev(nt, angle):
    k = np.arange(nt)
    vector = (1 / np.sqrt(nt)) * np.exp(-1j * np.pi * np.cos(angle) * k)
    return vector[:, np.newaxis]

def beamforming_vector_generation_P(P, nt):
    p = np.arange(P)
    cosp = (1 / np.pi) * np.angle(np.exp(1j * (2 * np.pi / P) * p))
    phi_p = np.arccos(cosp)
    F = np.zeros((nt, P), dtype=complex)
    for idx_p in range(P):
        F[:, idx_p] = np.squeeze(ev(nt, phi_p[idx_p]), -1)
    return F

def beamforming_vector_generation_Q(Q, nr):
    q = np.arange(Q)
    cosq = (1 / np.pi) * np.angle(np.exp(-1j * (2 * np.pi / Q) * q))
    phi_q = np.arccos(cosq)
    W = np.zeros((nr, Q), dtype=complex)
    for idx_q in range(Q):
        W[:, idx_q] = np.squeeze(ev(nr, phi_q[idx_q]), -1)
    return W

def generate_points(M_pts, delta, max_attempts=10000, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    points, attempts = [], 0
    while len(points) < M_pts and attempts < max_attempts:
        x = rng.uniform(0, math.pi); y = rng.uniform(0, math.pi)
        if not any(math.hypot(x - p[0], y - p[1]) < delta for p in points):
            points.append((x, y))
        attempts += 1
    if len(points) < M_pts:
        raise ValueError(f'Could not place {M_pts} points with delta={delta}')
    return points

def generate_noise(var_alpha, SNR, Q, P, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    var_noise = var_alpha * 10 ** (-SNR / 10)
    sigma_n = np.sqrt(var_noise / 2)
    return sigma_n * (rng.standard_normal((Q, P)) + 1j * rng.standard_normal((Q, P)))

def generate_channel_v2(nr, nt, angle_v, alpha_l):
    Lp = len(alpha_l)
    Hl = np.zeros((nr, nt, Lp), dtype=complex)
    for l in range(Lp):
        qq = ev(nt, angle_v[l]).conj().T
        ww = ev(nr, angle_v[l + Lp])
        Hl[:, :, l] = alpha_l[l] * (ww * qq)
    return np.sum(np.sqrt(nt * nr) * Hl, axis=-1)

def get_real_imag(H):
    return np.dstack((np.real(H), np.imag(H)))

print('Exact physics functions loaded.')

## Part 2 — Physics dictionary (Bug 3 + Bug 4 fix shoho)

- **Bug 3 fix**: U, V ke unit spectral norm-e rescale kora (Lipschitz constant = 1,
  tai ISTA step ~0.9 shothik ebong stable).
- **Bug 4 fix**: dictionary grid **omega-te uniform** (angle-e na) -- output-er
  shathe SHOMOI coordinate system-e thakar jonno.

In [ ]:
NT = NR = P_CB = Q_CB = 16
G_GRID = 32
L_MAX = 9          # generator-er L range (1..9) match kore

F_CB = beamforming_vector_generation_P(P_CB, NT)
W_CB = beamforming_vector_generation_Q(Q_CB, NR)

_om_u = np.linspace(0, 2 * np.pi, G_GRID, endpoint=False)
_om_phys = np.where(_om_u > np.pi, _om_u - 2 * np.pi, _om_u)
psis = np.arccos(np.clip(-_om_phys / np.pi, -1, 1))
phis = np.arccos(np.clip( _om_phys / np.pi, -1, 1))
A_r = np.hstack([ev(NR, a) for a in psis])
A_t = np.hstack([ev(NT, a) for a in phis])
U_dict = (W_CB.conj().T @ A_r).astype(np.complex64)
V_dict = (F_CB.T @ A_t.conj()).astype(np.complex64)
_sU = np.linalg.svd(U_dict, compute_uv=False)[0]
_sV = np.linalg.svd(V_dict, compute_uv=False)[0]
U_norm = (U_dict / _sU).astype(np.complex64)
V_norm = (V_dict / _sV).astype(np.complex64)
print(f'sigma_max(U)={_sU:.4f}  sigma_max(V)={_sV:.4f}  (normalized -> L=1)')

### Part 2.1 — Dictionary sanity check

In [ ]:
test_phi, test_psi = 1.1, 2.0
H_t = generate_channel_v2(NR, NT, np.array([test_phi, test_psi]), np.array([1.0+0j]))
G_t = W_CB.conj().T @ H_t @ F_CB
score = np.abs(U_dict.conj().T @ G_t @ V_dict.conj())
i_hat, j_hat = np.unravel_index(np.argmax(score), score.shape)
_tol_psi = 2 * np.max(np.diff(np.sort(psis)))
_tol_phi = 2 * np.max(np.diff(np.sort(phis)))
print(f'true  psi={test_psi:.3f} phi={test_phi:.3f}')
print(f'found psi={psis[i_hat]:.3f} phi={phis[j_hat]:.3f}')
ok = abs(psis[i_hat]-test_psi) < _tol_psi and abs(phis[j_hat]-test_phi) < _tol_phi
print('DICTIONARY CHECK:', 'PASS' if ok else 'FAIL -- STOP, kichu vul ache')

plt.figure(figsize=(4, 3.5))
plt.imshow(score, cmap='hot', origin='lower')
plt.scatter([j_hat], [i_hat], marker='x', c='cyan', s=80)
plt.title('Matched-filter score'); plt.xlabel('phi grid'); plt.ylabel('psi grid')
plt.tight_layout(); plt.show()

## Part 3 — Infinite training generator (Bug 1 fix)

Protiti sample fresh -- kokhono repeat hoy na. Angle-gulo `L_MAX=9` porjonto pad
kora hoy, ekta mask diye kon-gulo ashole ache seta bola thake.

In [ ]:
def pia_sample_generator():
    rng = np.random.default_rng()
    while True:
        Lp = int(rng.integers(1, 10))
        SNR = int(rng.integers(-15, 25))
        alpha_l = np.sqrt(1.0/Lp) * (rng.standard_normal(Lp)+1j*rng.standard_normal(Lp)) / np.sqrt(2)
        alpha_l = alpha_l[np.argsort(np.abs(alpha_l))[::-1]]
        pts = generate_points(Lp, np.pi/6, rng=rng)
        phi_l = np.array([p[0] for p in pts]); psi_l = np.array([p[1] for p in pts])
        angle_v = np.hstack([phi_l, psi_l])
        H = generate_channel_v2(NR, NT, angle_v, alpha_l)
        Gm = W_CB.conj().T @ H @ F_CB
        Z = generate_noise(1.0, float(SNR), Q_CB, P_CB, rng=rng)
        Y = Gm + Z
        data = get_real_imag(Y).astype(np.float32)
        psi_pad = np.zeros(L_MAX, dtype=np.float32); phi_pad = np.zeros(L_MAX, dtype=np.float32)
        mask = np.zeros(L_MAX, dtype=np.float32)
        psi_pad[:Lp] = psi_l; phi_pad[:Lp] = phi_l; mask[:Lp] = 1.0
        yield data, psi_pad, phi_pad, mask, np.float32(Lp)

_g = pia_sample_generator()
_d, _p, _f, _m, _L = next(_g)
print('sample: data', _d.shape, ' L =', int(_L), ' mask =', _m)

## Part 4 — Architecture: physics front-end + set-prediction head

- **Path A (physics)**: LearnedISTA (Bug 2/3/4 shoho fixed), age-r motoi
- **Path B (learned)**: raw observation-er upor shorashori CNN, 32x32-e upsample
- Duita fuse kore, **L_MAX=9-ta learned query** (DETR-style) cross-attention diye
  feature-er shathe mile, protita query theke shorashori (presence, sin/cos psi,
  sin/cos phi) ber hoy -- **kono spatial decoder/upsampling nei**।

In [ ]:
MAX_ISTA_VAL = 10.0
def _inv_softplus(v): return float(np.log(np.expm1(v)))

class LearnedISTA(tf.keras.layers.Layer):
    def __init__(self, U_dict, V_dict, n_iters=6, **kwargs):
        super().__init__(**kwargs)
        self.U = tf.constant(U_dict, dtype=tf.complex64)
        self.V = tf.constant(V_dict, dtype=tf.complex64)
        self.n_iters = n_iters
        self.G = U_dict.shape[1]

    def build(self, input_shape):
        self.s = [self.add_weight(name=f'step_{k}', shape=(), dtype=tf.float32,
                                   initializer=tf.keras.initializers.Constant(_inv_softplus(0.9)))
                  for k in range(self.n_iters)]
        self.t = [self.add_weight(name=f'thresh_{k}', shape=(), dtype=tf.float32,
                                   initializer=tf.keras.initializers.Constant(_inv_softplus(0.01)))
                  for k in range(self.n_iters)]

    def call(self, y_real_imag):
        Y = tf.complex(y_real_imag[..., 0], y_real_imag[..., 1])
        batch = tf.shape(Y)[0]
        nrm = tf.norm(tf.reshape(tf.abs(Y), (batch, -1)), axis=1)
        nrm = tf.reshape(nrm, (-1, 1, 1)) + 1e-9
        Yn = Y / tf.complex(nrm, tf.zeros_like(nrm))
        X = tf.zeros((batch, self.G, self.G), dtype=tf.float32)
        Uc = tf.math.conj(self.U); Vc = tf.math.conj(self.V)
        for k in range(self.n_iters):
            Xc = tf.cast(X, tf.complex64)
            Yhat = tf.einsum('qi,bij,pj->bqp', self.U, Xc, self.V)
            R = Yn - Yhat
            grad = tf.math.real(tf.einsum('qi,bqp,pj->bij', Uc, R, Vc))
            step = tf.nn.softplus(self.s[k]); th = tf.nn.softplus(self.t[k])
            X = tf.nn.relu(X + step * grad - th)
            X = tf.clip_by_value(X, 0.0, MAX_ISTA_VAL)
        return tf.expand_dims(X, axis=-1)


class QueryBroadcast(tf.keras.layers.Layer):
    """L_MAX-ta learned query embedding, batch-e broadcast kore."""
    def __init__(self, n_queries, d_model, **kwargs):
        super().__init__(**kwargs)
        self.n_queries = n_queries
        self.embed = tf.keras.layers.Embedding(n_queries, d_model)

    def call(self, ref):
        q = self.embed(tf.range(self.n_queries))
        q = tf.expand_dims(q, 0)
        return tf.tile(q, [tf.shape(ref)[0], 1, 1])


def build_set_regression_model(d_model=64, n_heads=4, n_queries=L_MAX, n_decoder_layers=2):
    inputs = tf.keras.Input(shape=(P_CB, Q_CB, 2), name='raw_observation')

    ista_map = LearnedISTA(U_norm, V_norm, n_iters=6, name='learned_ista')(inputs)  # (G,G,1)

    b = L.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    b = L.Conv2D(32, 3, padding='same', activation='relu')(b)
    b = L.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(b)  # 16->32
    b = L.Conv2D(32, 3, padding='same', activation='relu')(b)

    x = L.Concatenate()([ista_map, b])
    x = L.Conv2D(d_model, 3, padding='same', activation='relu')(x)
    x = L.Conv2D(d_model, 3, padding='same', activation='relu')(x)

    feat = L.Reshape((G_GRID * G_GRID, d_model))(x)     # (1024, d_model) -- keys/values
    q = QueryBroadcast(n_queries, d_model, name='query_embed')(feat)

    for _ in range(n_decoder_layers):
        attn = L.MultiHeadAttention(num_heads=n_heads, key_dim=d_model // n_heads)(q, feat, feat)
        q = L.LayerNormalization()(L.Add()([q, attn]))
        ff = L.Dense(d_model * 2, activation='relu')(q)
        ff = L.Dense(d_model)(ff)
        q = L.LayerNormalization()(L.Add()([q, ff]))

    presence = L.Dense(1, name='presence')(q)
    ang_raw = L.Dense(4, name='ang_raw')(q)
    outputs = L.Concatenate(axis=-1)([presence, ang_raw])   # (n_queries, 5)
    return tf.keras.Model(inputs, outputs, name='SetReg-PIA-Net')


model = build_set_regression_model()
model.summary()
SETREG_PARAM_COUNT = model.count_params()
print('PARAM COUNT:', f'{SETREG_PARAM_COUNT:,}')

## Part 5 — Hungarian-matching set-prediction loss (DETR-style)

- Cost = angular distance (na j matching-e presence-o thoda weight pay)
- Matched slot: circular regression loss `1 - cos(pred - true)` (psi ar phi duitar
  jonno) + presence target = 1
- Unmatched slot: presence target = 0, kintu **eos_coef diye down-weight kora** --
  DETR paper-er trick, karon beshi slot-i "no object" hoy (L=3 hole 9-er modhdhe
  6-ta), unweighted BCE oi majority-i dominate kore fele, "ache" slot-gulo shikhte
  deri hoy. eos_coef=0.4 diye "nei" slot-er loss-take kom weight deya hocche jate
  "ache" slot shekha-tai priority pay.

In [ ]:
EOS_COEF = 0.4   # DETR-er 'no-object' down-weighting trick

def angdiff(a, b):
    return np.angle(np.exp(1j * a) * np.exp(-1j * b))

def match_batch(pred_psi, pred_phi, pred_pres, true_psi, true_phi, true_mask):
    out = []
    B = pred_psi.shape[0]
    for b in range(B):
        Lp = int(true_mask[b].sum())
        if Lp == 0:
            out.append((np.array([], dtype=int), np.array([], dtype=int))); continue
        cost = np.zeros((L_MAX, Lp))
        for j in range(Lp):
            d_psi = angdiff(pred_psi[b], true_psi[b, j])
            d_phi = angdiff(pred_phi[b], true_phi[b, j])
            cost[:, j] = np.abs(d_psi) + np.abs(d_phi) - 0.3 * pred_pres[b]
        r, c = linear_sum_assignment(cost)
        out.append((r, c))
    return out


def compute_loss_and_grads(model, X, true_psi, true_phi, true_mask, alpha_ang=2.0, alpha_pres=1.0):
    with tf.GradientTape() as tape:
        out = model(X, training=True)
        presence_logit = out[..., 0]
        sin_psi, cos_psi = out[..., 1], out[..., 2]
        sin_phi, cos_phi = out[..., 3], out[..., 4]
        npsi = tf.sqrt(sin_psi**2 + cos_psi**2) + 1e-6
        nphi = tf.sqrt(sin_phi**2 + cos_phi**2) + 1e-6
        sin_psi_n, cos_psi_n = sin_psi/npsi, cos_psi/npsi
        sin_phi_n, cos_phi_n = sin_phi/nphi, cos_phi/nphi

        pred_pres_np = tf.sigmoid(presence_logit).numpy()
        pred_psi_np = np.arctan2(sin_psi_n.numpy(), cos_psi_n.numpy())
        pred_phi_np = np.arctan2(sin_phi_n.numpy(), cos_phi_n.numpy())

        matches = match_batch(pred_psi_np, pred_phi_np, pred_pres_np,
                              true_psi.numpy(), true_phi.numpy(), true_mask.numpy())

        pres_target = np.zeros((X.shape[0], L_MAX), dtype=np.float32)
        pres_weight = np.full((X.shape[0], L_MAX), EOS_COEF, dtype=np.float32)
        ang_loss = tf.constant(0.0); count = 0
        for b, (r, c) in enumerate(matches):
            if len(r) == 0:
                continue
            pres_target[b, r] = 1.0
            pres_weight[b, r] = 1.0     # matched ('present') slots keep full weight
            tp = tf.constant(true_psi[b].numpy()[c], dtype=tf.float32)
            tph = tf.constant(true_phi[b].numpy()[c], dtype=tf.float32)
            cp = tf.gather(cos_psi_n[b], r); sp = tf.gather(sin_psi_n[b], r)
            cph = tf.gather(cos_phi_n[b], r); sph = tf.gather(sin_phi_n[b], r)
            l = (1.0 - (cp*tf.cos(tp) + sp*tf.sin(tp))) + (1.0 - (cph*tf.cos(tph) + sph*tf.sin(tph)))
            ang_loss = ang_loss + tf.reduce_sum(l)
            count += len(r)
        ang_loss = ang_loss / max(count, 1)

        pres_bce = tf.nn.sigmoid_cross_entropy_with_logits(
            labels=tf.constant(pres_target), logits=presence_logit)
        pres_loss = tf.reduce_mean(pres_bce * tf.constant(pres_weight))

        total_loss = alpha_ang * ang_loss + alpha_pres * pres_loss

    grads = tape.gradient(total_loss, model.trainable_variables)
    return total_loss, ang_loss, pres_loss, grads


print('Loss function ready (eos_coef =', EOS_COEF, ')')

## Part 6 — Training

Custom loop (`model.fit` na) -- karon protiti step-e Hungarian matching (scipy,
numpy-level) lagbe, ja ekta static Keras loss function-e shorashori kora jay na.

Proti `PROBE_EVERY` step-e ekta chhoto held-out probe-e **argmax/oracle-Pd** print
kora hoy -- karon raw loss dekhe kichu bujha jay na (age-r shomporke shekha hoyeche)।

In [ ]:
BATCH = 32
STEPS_PER_EPOCH = 200
EPOCHS = 60                # 32*200*60 = 384,000 fresh sample -- PIA-Net-er scale-er kachakachi
PROBE_EVERY = 200          # protiti epoch sheshe probe

opt = tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=5.0)

# fixed probe set (generator theke ekbar-i banano, jate epoch-to-epoch tulonajogyo hoy)
_pg = pia_sample_generator()
_probe_X, _probe_psi, _probe_phi, _probe_mask = [], [], [], []
for _ in range(128):
    d, p, f, m, _ = next(_pg)
    _probe_X.append(d); _probe_psi.append(p); _probe_phi.append(f); _probe_mask.append(m)
_probe_X = np.stack(_probe_X); _probe_psi = np.stack(_probe_psi)
_probe_phi = np.stack(_probe_phi); _probe_mask = np.stack(_probe_mask)

def probe_oracle_pd(model, X, true_psi, true_phi, true_mask, max_deg=1.0):
    out = model(tf.constant(X), training=False).numpy()
    sin_psi, cos_psi = out[..., 1], out[..., 2]; sin_phi, cos_phi = out[..., 3], out[..., 4]
    npsi = np.sqrt(sin_psi**2+cos_psi**2)+1e-6; nphi = np.sqrt(sin_phi**2+cos_phi**2)+1e-6
    p_psi = np.arctan2(sin_psi/npsi, cos_psi/npsi); p_phi = np.arctan2(sin_phi/nphi, cos_phi/nphi)
    good, tot = 0, 0
    for i in range(len(X)):
        Lp = int(true_mask[i].sum())
        if Lp == 0: continue
        cost = np.zeros((Lp, L_MAX))
        for j in range(Lp):
            d1 = angdiff(true_psi[i, j], p_psi[i]); d2 = angdiff(true_phi[i, j], p_phi[i])
            cost[j] = np.abs(d1) + np.abs(d2)
        r, c = linear_sum_assignment(cost)
        for j, k in zip(r, c):
            e = (np.abs(np.degrees(angdiff(true_psi[i, j], p_psi[i, k]))) +
                np.abs(np.degrees(angdiff(true_phi[i, j], p_phi[i, k])))) / 2
            good += int(e < max_deg); tot += 1
    return good / max(tot, 1)


gen = pia_sample_generator()
history = {'total': [], 'ang': [], 'pres': [], 'oracle_pd': []}
t0 = time.time()
step_global = 0
for epoch in range(EPOCHS):
    ep_total, ep_ang, ep_pres = [], [], []
    for step in range(STEPS_PER_EPOCH):
        Xb, psib, phib, maskb = [], [], [], []
        for _ in range(BATCH):
            d, p, f, m, _ = next(gen)
            Xb.append(d); psib.append(p); phib.append(f); maskb.append(m)
        Xb = tf.constant(np.stack(Xb)); psib = tf.constant(np.stack(psib))
        phib = tf.constant(np.stack(phib)); maskb = tf.constant(np.stack(maskb))

        total, ang_l, pres_l, grads = compute_loss_and_grads(model, Xb, psib, phib, maskb)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        ep_total.append(float(total)); ep_ang.append(float(ang_l)); ep_pres.append(float(pres_l))
        step_global += 1
        if step % 20 == 0:
            # heartbeat: epoch-level print only comes after all 200 steps, which
            # can look like a hang -- this proves the loop is actually alive.
            print(f'  ...epoch {epoch+1} step {step:3d}/{STEPS_PER_EPOCH}  '
                  f'total={float(total):.4f}  ang={float(ang_l):.4f}  pres={float(pres_l):.4f}  '
                  f'({time.time()-t0:.0f}s elapsed)', flush=True)

    o_pd = probe_oracle_pd(model, _probe_X, _probe_psi, _probe_phi, _probe_mask)
    history['total'].append(np.mean(ep_total)); history['ang'].append(np.mean(ep_ang))
    history['pres'].append(np.mean(ep_pres)); history['oracle_pd'].append(o_pd)
    print(f'epoch {epoch+1:3d}/{EPOCHS}  total={np.mean(ep_total):.4f}  ang={np.mean(ep_ang):.4f}  '
          f'pres={np.mean(ep_pres):.4f}  oracle_Pd(probe)={o_pd:.4f}  ({time.time()-t0:.0f}s)', flush=True)

print(f'\nTraining shesh: {EPOCHS} epoch, {time.time()-t0:.0f}s, '
      f'{EPOCHS*STEPS_PER_EPOCH*BATCH:,} fresh samples')
model.save_weights('setreg_net.weights.h5')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(history['ang'], label='angular loss')
axs[0].plot(history['pres'], label='presence loss')
axs[0].set_xlabel('epoch'); axs[0].legend(); axs[0].grid(alpha=0.3); axs[0].set_title('training loss')
axs[1].plot(history['oracle_pd'], color='crimson')
axs[1].set_xlabel('epoch'); axs[1].set_ylabel('oracle Pd (held-out probe)')
axs[1].grid(alpha=0.3); axs[1].set_title('is the angle regression actually learning?')
plt.tight_layout(); plt.show()
print('oracle_Pd 0-theke spashto barche mane angle regression shikhche.')
print('flat/0-e atke thakle beshi epoch-eo eta shikhche na -- tokhon idea-ta punorbibechona lagbe.')

## Part 7 — Tomar real test set load koro (evaluation-er jonno)

In [ ]:
def find_dataset_dir(search_roots, markers=('test_data.npz', 'val_data.npz', 'train_data.npz')):
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if any(m in filenames for m in markers):
                return dirpath
    return None

DATA_DIR = find_dataset_dir(['/kaggle/input', '/content', '.'])
if DATA_DIR is None:
    DATA_DIR = '/kaggle/input/dl-doa/content/DL_DOA_CLONE/dataset'
print('DATA_DIR =', DATA_DIR)

def _p(*names):
    for n in names:
        q = os.path.join(DATA_DIR, n)
        if os.path.exists(q):
            return q
    return None

def _load_features(path):
    if path is None: return None
    if path.endswith('.pkl'):
        with open(path, 'rb') as f: return pickle.load(f)
    return np.load(path, allow_pickle=True)

tp, gp, mp = _p('test_data.npz'), _p('test_gt.npz'), _p('test_meta.npz')
if tp and gp and mp:
    X_test = np.load(tp)['data']; Y_test = np.load(gp)['data']; meta_test = np.load(mp)['data']
    feat_test = _load_features(_p('test_features.npy', 'test_features.pkl'))
    print('X_test:', X_test.shape, ' meta_test:', meta_test.shape)
else:
    X_test = Y_test = meta_test = feat_test = None
    print('test set paoa jayni.')

### Part 7.1 — 64x64 stored input theke raw 16x16 recover koro

In [ ]:
import scipy.ndimage as ndi

def build_recovery_index(raw_size=16, zoom_factor=4):
    idx_map = ndi.zoom(np.arange(raw_size), zoom_factor, order=0)
    return np.array([np.where(idx_map == i)[0][0] for i in range(raw_size)])

_recovery_idx = build_recovery_index(16, 4)
def recover_raw_batch(data_64):
    return data_64[:, _recovery_idx, :, :][:, :, _recovery_idx, :]

if X_test is not None:
    X_test_raw = recover_raw_batch(X_test)
    print('X_test_raw:', X_test_raw.shape)
    _chk = ndi.zoom(X_test_raw[0, :, :, 0], 4, order=0)
    print('recovery lossless:', np.allclose(_chk, X_test[0, :, :, 0]))
else:
    X_test_raw = None

## Part 8 — Evaluation (repo-r `TVT_Blob_Inference.py`-r metric function hubohu)

Model shorashori angle dey, tai blob detector-er dorkar nei -- shudhu `prepare_for_metric`
theke shuru।

In [ ]:
def wrap_2pi_to_minus_pi(a):
    a = np.asarray(a)
    return np.where(a > np.pi, a - 2 * np.pi, a)

def permute_pairs(A, B):
    A = np.asarray(A); B = np.asarray(B)
    d = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)
    r, c = linear_sum_assignment(d)
    return [(tuple(A[i]), tuple(B[j])) for i, j in zip(r, c)]

def prepare_for_metric(angles_est, feat):
    Lp = feat.shape[-1]
    if len(angles_est[0]) < Lp:
        return (np.array([feat[0], feat[1]]),
                np.array([np.full((Lp,), np.nan), np.full((Lp,), np.nan)]))
    ae = (angles_est[0][:Lp], angles_est[1][:Lp])
    perm = permute_pairs(list(zip(feat[0], feat[1])), list(zip(ae[0], ae[1])))
    psi_t, phi_t = zip(*[p[0] for p in perm])
    psi_e, phi_e = zip(*[p[1] for p in perm])
    return np.array([psi_t, phi_t]), np.array([psi_e, phi_e])

def get_ang_difference(gt_angles, pred_angles):
    H = np.angle(np.exp(1j * gt_angles) * np.exp(-1j * pred_angles))
    return (H * (180 / np.pi)).flatten()

def filter_angles(d, max_deg_error=1.0):
    return d[np.abs(d) <= max_deg_error], d[np.abs(d) > max_deg_error]

print('Metric utilities ready (identical logic to repo).')

In [ ]:
def evaluate_setreg(model, X_raw, meta, feat, batch=64):
    preds = model.predict(X_raw, batch_size=batch, verbose=0)
    presence = 1 / (1 + np.exp(-preds[..., 0]))
    sin_psi, cos_psi = preds[..., 1], preds[..., 2]
    sin_phi, cos_phi = preds[..., 3], preds[..., 4]
    npsi = np.sqrt(sin_psi**2+cos_psi**2)+1e-6; nphi = np.sqrt(sin_phi**2+cos_phi**2)+1e-6
    pred_psi = np.arctan2(sin_psi/npsi, cos_psi/npsi)
    pred_phi = np.arctan2(sin_phi/nphi, cos_phi/nphi)

    results = {}
    for i in range(len(X_raw)):
        Lp, SNR, QP = int(meta[i, 0]), int(meta[i, 1]), int(meta[i, 2])
        order = np.argsort(-presence[i])[:Lp]
        angles_est = (pred_psi[i, order], pred_phi[i, order])
        gt_a, pr_a = prepare_for_metric(angles_est, feat[i])
        results.setdefault((Lp, SNR, QP), []).append((gt_a, pr_a))

    rmse, pd_ = {}, {}
    for cond, ex in results.items():
        good, bad = [], []
        for gt_a, pr_a in ex:
            if np.isnan(pr_a).any():
                bad.append(np.full(gt_a.size, 999.0)); continue
            g, b = filter_angles(get_ang_difference(gt_a, pr_a), 1.0)
            good.append(g); bad.append(b)
        good = np.concatenate(good) if good else np.array([])
        bad = np.concatenate(bad) if bad else np.array([])
        tot = len(good) + len(bad)
        rmse[cond] = np.sqrt(np.mean(good**2)) if len(good) else np.nan
        pd_[cond] = len(good)/tot if tot else np.nan
    return rmse, pd_


if X_test_raw is not None:
    t0 = time.time()
    setreg_rmse, setreg_pd = evaluate_setreg(model, X_test_raw, meta_test, feat_test)
    print(f'evaluated {len(X_test_raw)} samples in {time.time()-t0:.1f}s')
    for c in sorted(setreg_rmse, key=lambda k: k[1]):
        print(c, f'RMSE={setreg_rmse[c]:.4f}', f'Pd={setreg_pd[c]:.4f}')
else:
    setreg_rmse, setreg_pd = {}, {}

## Part 9 — Hardcoded baseline reference (UNet / ResNet)

Repo-r committed, actually-run reproduction theke (`DL_DOA/figures_unet/*.pkl`,
`DL_DOA/figures_resnet/*.pkl`).

In [ ]:
UNET_REF_RMSE = {(3,-10,16):0.5498,(3,-5,16):0.5069,(3,0,16):0.4548,(3,5,16):0.3777,
                 (3,10,16):0.3047,(3,15,16):0.2556,(3,20,16):0.2297,(3,25,16):0.2086}
UNET_REF_PD   = {(3,-10,16):0.2226,(3,-5,16):0.4706,(3,0,16):0.6788,(3,5,16):0.8127,
                 (3,10,16):0.8883,(3,15,16):0.9244,(3,20,16):0.9439,(3,25,16):0.9509}
RESNET_REF_RMSE = {(3,-10,16):0.5532,(3,-5,16):0.5118,(3,0,16):0.4581,(3,5,16):0.3920,
                   (3,10,16):0.3256,(3,15,16):0.2792,(3,20,16):0.2528,(3,25,16):0.2377}
RESNET_REF_PD   = {(3,-10,16):0.2043,(3,-5,16):0.4366,(3,0,16):0.6396,(3,5,16):0.7790,
                   (3,10,16):0.8623,(3,15,16):0.8987,(3,20,16):0.9250,(3,25,16):0.9378}
UNET_PARAMS, RESNET_PARAMS = 31_276_481, 469_393
print('Reference table loaded.')

## Part 10 — THE HONEST TEST: real learning, na chance?

Test A: mean RMSE 0.5774 (uniform-error chance floor)-er niche kina.
Test B: SNR barle Pd barche kina (SNR-er shathe correlation).

In [ ]:
CHANCE_RMSE = 1.0 / np.sqrt(3)

if setreg_rmse:
    snrs_p = sorted(s for (l, s, q) in setreg_rmse if l == 3 and q == 16)
    r_vals = np.array([setreg_rmse[(3, s, 16)] for s in snrs_p], dtype=float)
    p_vals = np.array([setreg_pd[(3, s, 16)] for s in snrs_p], dtype=float)
    mean_rmse = np.nanmean(r_vals)
    testA = mean_rmse < 0.52
    valid = ~np.isnan(p_vals)
    corr = np.corrcoef(np.array(snrs_p)[valid], p_vals[valid])[0, 1] if valid.sum() > 2 else np.nan
    pd_rise = np.nanmax(p_vals) - np.nanmin(p_vals)
    testB = (corr > 0.7) and (pd_rise > 0.15)

    print('='*70)
    print('TEST A -- RMSE floor')
    print(f'  chance floor = {CHANCE_RMSE:.4f}   your mean RMSE = {mean_rmse:.4f}')
    print(f'  -> {"PASS" if testA else "FAIL"}')
    print()
    print('TEST B -- Pd rises with SNR')
    print(f'  corr(SNR, Pd) = {corr:.3f}   Pd range = {pd_rise:.3f}')
    print(f'  -> {"PASS" if testB else "FAIL"}')
    print('='*70)
    print('LEARNING' if (testA and testB) else ('PARTIAL' if testB else 'STILL AT CHANCE'))
    print('='*70)
else:
    print('no evaluation available.')

## Part 11 — Ceiling test: presence/selection kotota mullo pay?

Set-regression-e "blob clutter" thakar kotha na (fixed L_max slot), kintu tobu-o
mapo -- oracle-selection (ashol angle-er kachakachi slot bachai kora, cheating,
deploy kora jabe na) ar deployable top-L-by-presence-er modhdhe koto gap ache.

In [ ]:
def evaluate_setreg_oracle(model, X_raw, meta, feat, batch=64):
    preds = model.predict(X_raw, batch_size=batch, verbose=0)
    sin_psi, cos_psi = preds[..., 1], preds[..., 2]
    sin_phi, cos_phi = preds[..., 3], preds[..., 4]
    npsi = np.sqrt(sin_psi**2+cos_psi**2)+1e-6; nphi = np.sqrt(sin_phi**2+cos_phi**2)+1e-6
    pred_psi = np.arctan2(sin_psi/npsi, cos_psi/npsi)
    pred_phi = np.arctan2(sin_phi/nphi, cos_phi/nphi)

    results = {}
    for i in range(len(X_raw)):
        Lp, SNR, QP = int(meta[i, 0]), int(meta[i, 1]), int(meta[i, 2])
        true_psi, true_phi = feat[i][0], feat[i][1]
        cost = np.zeros((Lp, L_MAX))
        for j in range(Lp):
            d1 = angdiff(true_psi[j], pred_psi[i]); d2 = angdiff(true_phi[j], pred_phi[i])
            cost[j] = np.abs(d1) + np.abs(d2)
        r, c = linear_sum_assignment(cost)
        angles_est = (pred_psi[i, c], pred_phi[i, c])
        gt_a, pr_a = prepare_for_metric(angles_est, feat[i])
        results.setdefault((Lp, SNR, QP), []).append((gt_a, pr_a))

    rmse, pd_ = {}, {}
    for cond, ex in results.items():
        good, bad = [], []
        for gt_a, pr_a in ex:
            if np.isnan(pr_a).any():
                bad.append(np.full(gt_a.size, 999.0)); continue
            g, b = filter_angles(get_ang_difference(gt_a, pr_a), 1.0)
            good.append(g); bad.append(b)
        good = np.concatenate(good) if good else np.array([])
        bad = np.concatenate(bad) if bad else np.array([])
        tot = len(good) + len(bad)
        rmse[cond] = np.sqrt(np.mean(good**2)) if len(good) else np.nan
        pd_[cond] = len(good)/tot if tot else np.nan
    return rmse, pd_


if X_test_raw is not None:
    orc_rmse, orc_pd = evaluate_setreg_oracle(model, X_test_raw, meta_test, feat_test)
    print(f'{"SNR":>5} | {"Pd top-L":>9} {"Pd ORACLE":>10} | {"headroom":>9}')
    for c in sorted(setreg_pd, key=lambda k: k[1]):
        o = orc_pd.get(c, float('nan'))
        print(f'{c[1]:>5} | {setreg_pd[c]:>9.4f} {o:>10.4f} | {o-setreg_pd[c]:>+9.4f}')
    mt = np.nanmean(list(setreg_pd.values())); mo = np.nanmean(list(orc_pd.values()))
    print(f'\nmean Pd top-L = {mt:.4f}   mean Pd oracle = {mo:.4f}   headroom = {mo-mt:+.4f}')
    print('boro headroom mane presence/selection-i ekhono dhire shikhche -- ang regression thik ache.')
    print('choto headroom mane angle regression-i main bottleneck.')

## Part 12 — Comparison plots vs the hardcoded baselines

In [ ]:
snrs = sorted(set(k[1] for k in UNET_REF_RMSE))
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))

axs[0].plot(snrs, [UNET_REF_RMSE[(3,s,16)] for s in snrs], 'o-', label='UNet (reference)')
axs[0].plot(snrs, [RESNET_REF_RMSE[(3,s,16)] for s in snrs], 's-', label='ResNet (reference)')
if setreg_rmse:
    ps = sorted(s for (l,s,q) in setreg_rmse if l==3 and q==16)
    axs[0].plot(ps, [setreg_rmse[(3,s,16)] for s in ps], '^-', color='crimson', label='SetReg-Net (this run)')
axs[0].axhline(CHANCE_RMSE, ls='--', c='gray', label='chance floor (0.577)')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('RMSE (deg)'); axs[0].set_title('RMSE vs SNR')
axs[0].legend(); axs[0].grid(alpha=0.3)

axs[1].plot(snrs, [UNET_REF_PD[(3,s,16)] for s in snrs], 'o-', label='UNet (reference)')
axs[1].plot(snrs, [RESNET_REF_PD[(3,s,16)] for s in snrs], 's-', label='ResNet (reference)')
if setreg_pd:
    ps = sorted(s for (l,s,q) in setreg_pd if l==3 and q==16)
    axs[1].plot(ps, [setreg_pd[(3,s,16)] for s in ps], '^-', color='crimson', label='SetReg-Net (this run)')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Pd'); axs[1].set_ylim(-0.02, 1.02)
axs[1].set_title('Detection probability vs SNR'); axs[1].legend(); axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
names = ['UNet', 'ResNet', 'SetReg-Net']
params = [UNET_PARAMS, RESNET_PARAMS, SETREG_PARAM_COUNT]
bars = ax.bar(names, params, color=['#4c72b0', '#dd8452', '#c44e52'])
ax.set_yscale('log'); ax.set_ylabel('Parameters (log)'); ax.set_title('Model size')
for b, p in zip(bars, params):
    ax.text(b.get_x()+b.get_width()/2, p, f'{p:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

## Part 13 — Final honest summary

In [ ]:
if setreg_rmse:
    print('='*74)
    print(f'{"SNR":>5} | {"SetReg RMSE":>11} {"SetReg Pd":>10} | {"UNet Pd":>8} {"ResNet Pd":>10} | {"gap-U":>7} {"gap-R":>7}')
    print('-'*74)
    gaps_u, gaps_r = [], []
    for s in sorted(s for (l,s,q) in setreg_pd if l==3 and q==16):
        c = (3, s, 16)
        gu = UNET_REF_PD[c] - setreg_pd[c]; gr = RESNET_REF_PD[c] - setreg_pd[c]
        gaps_u.append(gu); gaps_r.append(gr)
        print(f'{s:>5} | {setreg_rmse[c]:>11.3f} {setreg_pd[c]:>10.3f} | {UNET_REF_PD[c]:>8.3f} '
              f'{RESNET_REF_PD[c]:>10.3f} | {gu:>+7.3f} {gr:>+7.3f}')
    print('='*74)
    print(f'Mean Pd gap vs UNet:   {np.nanmean(gaps_u):+.3f}')
    print(f'Mean Pd gap vs ResNet: {np.nanmean(gaps_r):+.3f}')
    print(f'Model size: SetReg-Net {SETREG_PARAM_COUNT:,} vs ResNet {RESNET_PARAMS:,}')
    print(f'Fresh training samples seen: {EPOCHS*STEPS_PER_EPOCH*BATCH:,}')
    print()
    print('Ei formulation-er dabi: heatmap-decoder/blob-detector-er upor nirbhor na kore')
    print('shorashori angle regression -- pixel quantization ar clutter structurally thakte pare na.')
    print('Number-ta ki bole segulo UPORE-e dekha jachche -- overclaim na kore.')
else:
    print('no evaluation available.')